In [1]:
!pip install cake-ensemble

In [2]:
import time
import numpy as np
from scipy.stats import pearsonr
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from cake_ensemble import sil_samples

# 1. Generate a larger synthetic dataset (10,000 points)
X, _ = make_blobs(n_samples=10000, centers=3, cluster_std=1.5, random_state=42)

# 2. Run KMeans once to get labels and centers
km = KMeans(n_clusters=3, n_init=1, random_state=42).fit(X)
labels, centers = km.labels_, km.cluster_centers_

print(f"Comparing Silhouette computation for N={len(X)} points...")

# 3. Time the exact computation (scikit-learn)
start = time.time()
sil_exact = sil_samples(X, labels, approximation=False)
time_exact = time.time() - start

# 4. Time the CAKE approximation (optimized O(N))
# Requires cluster centers, which are readily available from KMeans
start = time.time()
sil_approx = sil_samples(X, labels, approximation=True, centers=centers)
time_approx = time.time() - start

# 5. Calculate Pearson correlation
corr, _ = pearsonr(sil_exact, sil_approx)

# Output results
print(f"\n=== Performance & Correlation ====")
print(f"Exact Time (sklearn): {time_exact:.4f} seconds")
print(f"Approx Time (CAKE):   {time_approx:.4f} seconds")
print(f"Speedup factor:       {time_exact / time_approx:.1f}x")
print(f"Pearson Correlation:  {corr:.6f}")

Comparing Silhouette computation for N=10000 points...

=== Performance & Correlation ====
Exact Time (sklearn): 1.4867 seconds
Approx Time (CAKE):   0.0025 seconds
Speedup factor:       601.7x
Pearson Correlation:  0.955806
